In [16]:
from PIL import Image
import torch
import torch.utils.data
from torch.utils.data import Dataset, DataLoader
from torchvision import models, datasets, tv_tensors
from torchvision.transforms import v2
from torch import nn
import numpy as np

# 1. Image Preprocessing

In [17]:
labelPath = "top_potsdam_7_9_label.tif"
imgPath = "top_potsdam_7_9_RGB.tif"
valPath = "top_potsdam_7_8_RGB.tif"
valLabelPath = "top_potsdam_7_8_label.tif"
img = Image.open(imgPath)
label = Image.open(labelPath)
valImg = Image.open(imgPath)
valLabel = Image.open(labelPath)

In [18]:
transforms = v2.Compose(
    [
        v2.ToImage(),
    ]
)

In [19]:
imgTransformed, labelTransformed = transforms((img, label))
valImgTransformed, valLabelTransformed = transforms((valImg, valLabel))

In [20]:
colorDict = {}
i = 0
label2D = labelTransformed.reshape(3, -1).transpose(0,1)
for color in label2D.unique(dim=0):
    colorDict[color] = i
    i += 1

indexMap = torch.zeros(label_transformed.shape[0])
for color, index in colorDict.items():
    indexMap += torch.where(torch.all(label_transformed == color, dim=1), index, 0)
labels = nn.functional.one_hot(indexMap.to(torch.int64))

# 2. Dataset Setup

In [21]:
N = 200
valN = 20
targetHeight = 572
targetWidth = 572
datasetTransformation = v2.Compose(
    [
        v2.RandomCrop(targetHeight, targetWidth),
        v2.RandomHorizontalFlip(0.5),
    ]
)
inputTransformation = v2.Compose(
    [
        v2.ToDtype(torch.float32, scale=True),
        v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229,0.224,0.225]),
    ]
)
dataset = torch.zeros(N, 2, imgTransformed.shape[0], targetHeight, targetWidth)
valDataset = torch.zeros(N, 2, valImgTransformed.shape[0], targetHeight, targetWidth)
for i in range(N):
    sampleImg, sampleLabel = datasetTransformation(imgTransformed, labelTransformed)
    valSampleImg, valSampleLabel = datasetTransformation(valImgTransformed, valLabelTransformed)
    sampleImg = inputTransformation(sampleImg)
    valSampleImg = inputTransformation(valSampleImg)
    dataset[i, 0] = sampleImg
    valDataset[i, 0] = valSampleImg
    dataset[i, 1] = sampleLabel
    valDataset[i, 1] = valSampleLabel

In [22]:
class ImageDataset(Dataset):
    def __init__(self, dataset):
        self.data = dataset
        self.len = dataset.shape[0]
    def __getitem__(self, idx):
        return self.data[idx]
    def __len__(self):
        return self.len

In [23]:
train_dataset = ImageDataset(dataset)
val_dataset = ImageDataset(valDataset)

In [24]:
TRAIN_BATCH_SIZE = 1
VAL_BATCH_SIZE = 1
TEST_BATCH_SIZE = 1
LEARNING_RATE = 1e-04
EPOCHS = 10
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [25]:
train_loader = DataLoader(train_dataset, batch_size = TRAIN_BATCH_SIZE, shuffle = True, num_workers = 0, drop_last=True)
val_loader = DataLoader(valDataset, batch_size = VAL_BATCH_SIZE, shuffle = True, num_workers = 0, drop_last=True)

# 3. Model Setup

In [26]:
class UNet(nn.Module):
    def __init__(self, image_size, min_size, input_channels, base_channels, out_length):
        super().__init__()
        self.contracting = nn.ModuleList()
        self.contracting.append(nn.ModuleList(nn.Sequential(nn.Conv2d(input_channels, base_channels, kernel_size=(3, 3), padding=0), nn.ReLU(), nn.Conv2d(base_channels, base_channels, kernel_size=(3,3), padding = 0), nn.ReLU())))
        current_size = (image_size - 4) / 2
        current_channels = base_channels
        layer_count = 1
        while current_size != min_size:
            current_channels *= 2
            self.contracting.append(nn.ModuleList(nn.Sequential(nn.MaxPool2d(kernel_size = (2, 2), stride = 2), nn.Conv2d(current_channels // 2, current_channels, kernel_size=(3, 3), padding=0), nn.ReLU(), nn.Conv2d(current_channels, current_channels, kernel_size=(3,3), padding = 0), nn.ReLU())))
            current_size = (current_size - 4) / 2
            layer_count += 1
        current_channels *= 2
        self.contracting.append(nn.ModuleList(nn.Sequential(nn.MaxPool2d(kernel_size = (2, 2), stride = 2), nn.Conv2d(current_channels // 2, current_channels, kernel_size=(3, 3), padding=0), nn.ReLU(), nn.Conv2d(current_channels, current_channels, kernel_size=(3,3), padding = 0), nn.ReLU())))
        self.expanding = nn.ModuleList()
        self.expanding.append(nn.ModuleList(nn.Sequential(nn.ConvTranspose2d(current_channels, current_channels, kernel_size=(3,3), padding=0), nn.ReLU(), nn.ConvTranspose2d(current_channels, current_channels, kernel_size=(3,3), padding=0), nn.ReLU())))
        while layer_count > 0:
            current_channels //= 2
            self.expanding.append(nn.ModuleList(nn.Sequential(nn.Conv2d(current_channels*2, current_channels, kernel_size=(2,2), padding='same'),nn.ConvTranspose2d(current_channels * 2, current_channels, kernel_size=(3,3), padding = 0), nn.ReLU(), nn.ConvTranspose2d(current_channels, current_channels, kernel_size=(3,3), padding = 0), nn.ReLU())))
            layer_count -= 1
        #TODO: determine if output channels == number of classes
        self.expanding.append(nn.ModuleList(nn.Sequential(nn.Conv2d(current_channels, out_length, kernel_size = (1, 1)))))
        self.skip_list = []
    def forward(self, x):
        for model_list in self.contracting:
            for block in model_list:
                x = block(x)
            self.skip_list.append(x)
        self.skip_list.pop()
        for i, model_list in enumerate(self.expanding):
            if len(model_list) > 1 and i > 0:
                x = nn.Upsample(scale_factor=2, mode='bilinear')(x)
            x = model_list[0](x)
            if len(model_list) > 1 and i > 0:
                #crop_size = (self.skip_list[-i].shape[-1] - x.shape[-1]) // 2
                #TODO: check crop size for off by 1 errors
                x = torch.cat((self.skip_list[-i], x), dim = 1)
            for block in model_list[1:]:
                x = block(x)
        return x

In [27]:
model = UNet(572, 32, dataset.shape[2], 64, len(colorDict.keys()))
model = model.to(device)

# 4. Model Training

In [28]:
lossFunction = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(params = model.parameters(), lr = LEARNING_RATE)

In [29]:
def train(model, epochs, lossFunction, learningRate, verbose = True, scheduler = None, optimizer = None):
    if optimizer is None:
        optimizer = torch.optim.AdamW(params = model.parameters(), lr = learningRate)
    losses = []
    for epoch in range(epochs):
        model.train()
        aggregatedLoss = 0
        stepsPerPrint = 5
        epochsPerValidation = 1
        for batch, data in enumerate(train_loader):
            img = data[:, 0]
            label = data[:, 1].to(torch.int64)
            label2D = label.reshape(3, -1).transpose(0,1)
            indexMap = torch.zeros(label2D.shape[0])
            for color, index in colorDict.items():
                indexMap += torch.where(torch.all(label2D == color, dim=1), index, 0)
            label2DOHE = nn.functional.one_hot(indexMap.to(torch.int64), num_classes=len(colorDict.keys())).transpose(0, 1).reshape(1, len(colorDict.keys()), targetHeight, targetWidth).to(torch.float16)
            img = img.to(device)
            label2DOHE = label2DOHE.to(device)
            optimizer.zero_grad()
            output = model(img).squeeze(1)
            loss = lossFunction(output, label2DOHE)
            print(loss.item())
            aggregatedLoss += loss.item()
            loss.backward()
            optimizer.step()
            if batch % stepsPerPrint == stepsPerPrint-1 and not batch == 0:
                if verbose:
                    print(f"Epoch: {epoch}, Batch: {batch // TRAIN_BATCH_SIZE}, Loss: {aggregatedLoss / (stepsPerPrint)}")
                aggregatedLoss = 0
        if epoch % epochsPerValidation == 0:
            model.eval()
            with torch.inference_mode():
                valAggregatedLoss = 0
                valAggregatedMeanIoU = 0
                steps = 0
                iouSteps = 0
                for batch, data in enumerate(val_loader):
                    img = data[:, 0]
                    label = data[:, 1].to(torch.int64)
                    label2D = label.reshape(3, -1).transpose(0,1)
                    indexMap = torch.zeros(label2D.shape[0])
                    for color, index in colorDict.items():
                        indexMap += torch.where(torch.all(label2D == color, dim=1), index, 0)
                    label2DOHE = nn.functional.one_hot(indexMap.to(torch.int64), num_classes=len(colorDict.keys())).transpose(0, 1).reshape(1, len(colorDict.keys()), targetHeight, targetWidth).to(torch.float16)
                    img = img.to(device)
                    label2DOHE = label2DOHE.to(device)
                    output = model(img).squeeze(1)
                    loss = lossFunction(output, label2DOHE)
                    valAggregatedLoss += loss.item()
                    for color, index in colorDict.items():
                        groundTruthMap = torch.all(label2D == color, dim=1)
                        predClassMap = torch.argmax(output, dim = 1).reshape(1, -1) == index
                        sumMap = groundTruthMap.to(torch.int) + predClassMap.to(torch.int)
                        andMap = predClassMap & groundTruthMap
                        truePositive = torch.sum(andMap)
                        falsePositive = torch.sum(sumMap - groundTruthMap.to(torch.int))
                        falseNegative = torch.sum(sumMap - predClassMap.to(torch.int))
                        if truePositive:
                            valAggregatedMeanIoU += truePositive / (truePositive + falsePositive + falseNegative)
                            iouSteps += 1
                    steps += 1
                print(f"Average Dev Loss: {valAggregatedLoss / steps}")
                print(f"Average IoU Loss: {valAggregatedMeanIoU / iouSteps}")
    return losses

In [30]:
train(model, EPOCHS, lossFunction, LEARNING_RATE, optimizer = optimizer)

1.7727383375167847
1.7582874298095703
1.8613511323928833
1.7267625331878662
1.810678243637085
Epoch: 0, Batch: 4, Loss: 1.785963535308838
1.826549768447876
1.6179916858673096
1.819406270980835
1.7401105165481567
1.7549843788146973
Epoch: 0, Batch: 9, Loss: 1.751808524131775
1.8123265504837036
1.781177043914795
1.774226427078247
1.7314708232879639
1.722787857055664
Epoch: 0, Batch: 14, Loss: 1.7643977403640747
1.7037895917892456
1.83831787109375
1.8662546873092651
1.7960381507873535
1.456545114517212
Epoch: 0, Batch: 19, Loss: 1.7321890830993651
1.6256462335586548
1.8055864572525024
1.5674339532852173
1.2575243711471558
1.6912474632263184
Epoch: 0, Batch: 24, Loss: 1.5894876956939696
1.5806483030319214
3.1953155994415283
1.9972996711730957
1.9067059755325317
1.8164427280426025
Epoch: 0, Batch: 29, Loss: 2.099282455444336
1.2998478412628174
1.3829201459884644
1.7711362838745117
1.6715385913848877
1.7694634199142456
Epoch: 0, Batch: 34, Loss: 1.5789812564849854
1.7965441942214966
1.762128

KeyboardInterrupt: 